## Simulating Two Remote Nodes on Localhost

This notebook simulates two independent machines — a **local node**
and a **remote node** — both running on `127.0.0.1`.  They peer
over WebSockets and can transparently operate on each other's
policies as if they were on separate hosts.

### What this notebook does
1. Create a **local node** and a **remote node** (both on localhost).
2. Store an entry on the remote node.
3. Peer the two nodes over localhost WebSockets.
4. From the local node, switch to the remote node's policy via its proxy.
5. Query the remote node's memory — verify the entry is accessible.
6. Demonstrate bidirectional access: the remote node queries the local node.
7. Switch back to the local node and clean up.

In [ ]:
import time
import laila
from laila.macros.defaults import DefaultPolicy, DefaultPool, DefaultTCPIPProtocol

### 1 — Create a local node and a remote node

In [ ]:
local_node = DefaultPolicy()
remote_node = DefaultPolicy()

local_tcp = DefaultTCPIPProtocol()
remote_tcp = DefaultTCPIPProtocol()

laila.active_policy = local_node
laila.communication.add_connection(local_tcp)

laila.active_policy = remote_node
laila.communication.add_connection(remote_tcp)

print("Local node: ", local_node.global_id)
print("Remote node:", remote_node.global_id)

### 2 — Store an entry on the remote node

In [ ]:
laila.active_policy = remote_node

remote_pool = DefaultPool()
laila.memory.add_pool(pool=remote_pool, pool_nickname="remote-store")

entry = laila.constant(data={"message": "stored on the remote node"}, nickname="remote-entry")
with laila.guarantee:
    laila.memorize(entries=entry, pool_nickname="remote-store")

print("Entry:", entry.global_id)
print("Remote pool has entry:", entry.global_id in remote_pool.resource)

### 3 — Peer the two nodes over localhost

In [ ]:
remote_node.central.communication.start()

laila.active_policy = local_node
laila.communication.add_tcpip_peer("127.0.0.1", remote_tcp.port, remote_tcp.peer_secret_key)
time.sleep(0.3)

print("Peered successfully.")
print(f"Local peers:  {list(local_node.central.communication.peers)}")
print(f"Remote peers: {list(remote_node.central.communication.peers)}")

### 4 — Switch to the remote node's policy

Setting `laila.active_policy` to the remote proxy means all
subsequent `laila.*` calls route to the remote node over WebSockets.

In [ ]:
laila.active_policy = local_node
print("Active policy:", laila.active_policy.global_id, "(local)")

remote_proxy = laila.peers[remote_node.global_id]
laila.active_policy = remote_proxy
print("Active policy:", laila.active_policy.global_id, "(remote via proxy)")

### 5 — Query the remote node's memory

The proxy transparently traverses the remote node's policy tree
over the wire.

In [ ]:
routed_pool = remote_proxy.central.memory.pool_router.route(
    entries=[entry.global_id], pool_nickname="remote-store"
)
print("Remote route returned:", type(routed_pool).__name__)
print("Pool data (serialized via RPC):")
for k, v in routed_pool.items():
    if k == "resource":
        print(f"  resource: {len(v)} entries stored")
    else:
        print(f"  {k}: {v}")

### 6 — Bidirectional: the remote node queries the local node

Peering is symmetric — either node can call into the other.

In [ ]:
laila.active_policy = local_node

local_pool = DefaultPool()
laila.memory.add_pool(pool=local_pool, pool_nickname="local-store")

# From the remote node's perspective, call into the local node via proxy
laila.active_policy = remote_node
local_proxy = laila.peers[local_node.global_id]
local_routed = local_proxy.central.memory.pool_router.route(
    entries=["any-id"], pool_nickname="local-store"
)
print("Remote node queried local node's pool router via RPC.")
print("Local pool batch_accelerated:", local_routed.get("batch_accelerated"))

### 7 — Switch back to the local node and clean up

In [ ]:
laila.active_policy = local_node
print("Active policy:", laila.active_policy.global_id, "(local)")

local_node.central.communication.stop()
remote_node.central.communication.stop()
print("Connections closed.")